# 📡 Phase 3 – Model Monitoring with Prometheus & Grafana
**Goal:** Simulate a live prediction service that exposes metrics scraped by Prometheus and visualized in Grafana.

Topics:
- Exposing custom ML metrics via prometheus_client
- Tracking prediction latency, drift scores, accuracy over time
- Viewing live dashboards in Grafana (http://localhost:3000)

In [ ]:
import time
import random
import numpy as np
import threading
from prometheus_client import start_http_server, Counter, Histogram, Gauge, Summary
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
print('Libraries ready!')

In [ ]:
# Train a model to simulate serving
X, y = make_classification(n_samples=3000, n_features=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f'Model ready. Baseline accuracy: {model.score(X_test, y_test):.4f}')

In [ ]:
# Define Prometheus metrics
PREDICTION_COUNT    = Counter('ml_predictions_total', 'Total predictions made', ['result'])
PREDICTION_LATENCY  = Histogram('ml_prediction_latency_seconds', 'Prediction latency',
                                 buckets=[0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.5])
MODEL_ACCURACY      = Gauge('ml_model_accuracy', 'Rolling model accuracy (last 100 predictions)')
DRIFT_SCORE         = Gauge('ml_feature_drift_score', 'Simulated feature drift score')
CONFIDENCE_SUMMARY  = Summary('ml_prediction_confidence', 'Model prediction confidence scores')

# Start metrics server on port 8888/metrics (Prometheus scrapes this)
start_http_server(8888)
print('Prometheus metrics server started on :8888/metrics')
print('Check Prometheus at http://localhost:9090')
print('Grafana dashboard at http://localhost:3000 (admin/ailab123)')

In [ ]:
# Simulate live prediction traffic for 2 minutes
# Run this cell — it will stream metrics to Prometheus
print('Simulating prediction traffic... (runs for 120 seconds)')
print('Go check Grafana/Prometheus NOW to see live metrics!')

recent_correct = []
start_time = time.time()
run_duration = 120  # seconds
request_num = 0

while time.time() - start_time < run_duration:
    # Sample a random test instance
    idx = random.randint(0, len(X_test) - 1)
    x_sample = X_test[idx].reshape(1, -1)
    true_label = y_test[idx]
    
    # Simulate prediction with latency tracking
    t0 = time.time()
    proba = model.predict_proba(x_sample)[0]
    pred = np.argmax(proba)
    latency = time.time() - t0
    
    # Emit metrics
    PREDICTION_LATENCY.observe(latency)
    PREDICTION_COUNT.labels(result='correct' if pred == true_label else 'wrong').inc()
    CONFIDENCE_SUMMARY.observe(float(np.max(proba)))
    
    # Rolling accuracy
    recent_correct.append(int(pred == true_label))
    if len(recent_correct) > 100:
        recent_correct.pop(0)
    MODEL_ACCURACY.set(np.mean(recent_correct))
    
    # Simulate drift growing over time
    elapsed = time.time() - start_time
    drift = 0.05 + 0.15 * (elapsed / run_duration) + random.gauss(0, 0.01)
    DRIFT_SCORE.set(max(0, drift))
    
    request_num += 1
    if request_num % 100 == 0:
        print(f'  [{elapsed:.0f}s] Requests: {request_num}, Accuracy: {np.mean(recent_correct):.3f}, Drift: {drift:.3f}')
    
    time.sleep(0.05)  # ~20 RPS

print(f'\nDone! Total requests simulated: {request_num}')

## Grafana Setup
1. Open **http://localhost:3000** → Login: `admin` / `ailab123`
2. Go to **Explore** → Select **Prometheus** datasource
3. Try these PromQL queries:

```promql
# Total predictions per second
rate(ml_predictions_total[1m])

# P95 prediction latency
histogram_quantile(0.95, rate(ml_prediction_latency_seconds_bucket[5m]))

# Live model accuracy
ml_model_accuracy

# Drift score over time
ml_feature_drift_score
```

4. Create a **Dashboard** with panels for each metric!

**Exercise:** Add an alert rule in Grafana to fire when `ml_feature_drift_score > 0.15`!